In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE, SelectKBest, f_regression, mutual_info_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from sklearn.preprocessing import MinMaxScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set plotting style

In [2]:
plt.style.use('default')
sns.set_palette("husl")

print("=== UK STOCK MARKET ANALYSIS - MSc DATA SCIENCE PROJECT ===")
print("Author: [Vennela Yaganti]")
print("Supervisor: [Dr.Ala Barzinji]")
print("Course & College: MSc Data Science - University of Greenwich")
print("="*60)

=== UK STOCK MARKET ANALYSIS - MSc DATA SCIENCE PROJECT ===
Author: [Vennela Yaganti]
Supervisor: [Dr.Ala Barzinji]
Course & College: MSc Data Science - University of Greenwich


# Load the actual dataset

In [3]:
try:
    print("Loading UK Stock Market dataset...")
    df = pd.read_csv('/content/UK Stock Market 1988-2024.csv')
    print(f"✓ Dataset loaded successfully!")
    print(f"✓ Original dataset shape: {df.shape}")
    print(f"✓ Total entries: {df.shape[0]:,}")
    print(f"✓ Total columns: {df.shape[1]}")

    # Always reduce to 1000 rows for faster modeling
    print("\nReducing dataset size to 1000 rows for efficient modeling...")

    if df.shape[0] > 1000:
        df = df.sample(n=1000, random_state=42).reset_index(drop=True)
        print(f"✓ Dataset reduced to {df.shape[0]:,} records.")

except FileNotFoundError:
    print("❌ Dataset file 'UK Stock Market 1988-2024.csv' not found in current directory.")
    print("Please ensure the file is available in the working directory.")
    exit()


Loading UK Stock Market dataset...
✓ Dataset loaded successfully!
✓ Original dataset shape: (4403213, 10)
✓ Total entries: 4,403,213
✓ Total columns: 10

Reducing dataset size to 1000 rows for efficient modeling...
✓ Dataset reduced to 1,000 records.


# Display basic information

In [4]:
print("\nDataset Structure:")
print(df.info())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nBasic Statistics:")
print(df.describe())


Dataset Structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    1000 non-null   int64  
 1   Date          1000 non-null   object 
 2   Ticker        1000 non-null   object 
 3   Company_Name  999 non-null    object 
 4   Open          1000 non-null   float64
 5   High          1000 non-null   float64
 6   Low           1000 non-null   float64
 7   Close         1000 non-null   float64
 8   Adj Close     1000 non-null   float64
 9   Volume        1000 non-null   float64
dtypes: float64(6), int64(1), object(3)
memory usage: 78.3+ KB
None

Column Names:
['Unnamed: 0', 'Date', 'Ticker', 'Company_Name', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

First 5 rows:
   Unnamed: 0        Date  Ticker                               Company_Name  \
0     1801848  2013-01-29  HPAC.L    HERMES PACIFIC INVESTMENTS PLC ORD GBP1

# 1.1 DATA CLEANING AND PREPROCESSING

In [5]:
print("\n1.1 DATA CLEANING AND PREPROCESSING")
print("-" * 35)

# Handle index column if present
index_cols = [col for col in df.columns if 'unnamed' in col.lower() or col.lower() in ['index', 'id']]
if index_cols:
    df = df.drop(index_cols, axis=1)
    print(f"✓ Dropped index columns: {index_cols}")

# Standardize column names
df.columns = df.columns.str.strip().str.replace(' ', '_')
print("✓ Standardized column names")



1.1 DATA CLEANING AND PREPROCESSING
-----------------------------------
✓ Dropped index columns: ['Unnamed: 0']
✓ Standardized column names


# Identify date column

In [ ]:
date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
if date_cols:
    date_col = date_cols[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    print(f"✓ Converted {date_col} to datetime format")
else:
    print("⚠️ No date column found")

# Identify numeric columns (stock prices and volume)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"✓ Identified numeric columns: {numeric_cols}")

# Check for missing values

In [ ]:
missing_values = df.isnull().sum()
print(f"\nMissing Values Summary:")
for col, missing in missing_values.items():
    if missing > 0:
        print(f"  {col}: {missing} ({missing/len(df)*100:.2f}%)")

# Handle missing values

In [ ]:
if missing_values.sum() > 0:
    print("\nHandling missing values...")

    # For numeric columns, use forward fill then backward fill
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')

    # For categorical columns, use mode
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown')

    print("✓ Missing values handled")

# Remove duplicates

In [ ]:
initial_shape = df.shape[0]
df = df.drop_duplicates()
print(f"✓ Removed {initial_shape - df.shape[0]} duplicate rows")

# Data validation for stock data
price_cols = [col for col in df.columns if any(price_term in col.lower()
              for price_term in ['open', 'high', 'low', 'close', 'price'])]

if len(price_cols) >= 4:  # If we have OHLC data
    # Ensure High >= Low and other logical constraints
    high_col = [col for col in price_cols if 'high' in col.lower()]
    low_col = [col for col in price_cols if 'low' in col.lower()]

    if high_col and low_col:
        invalid_data = df[df[high_col[0]] < df[low_col[0]]]
        if len(invalid_data) > 0:
            print(f"⚠️ Found {len(invalid_data)} rows with High < Low, removing...")
            df = df[df[high_col[0]] >= df[low_col[0]]]

# Remove negative prices if any

In [ ]:
for col in price_cols:
    negative_prices = df[df[col] < 0]
    if len(negative_prices) > 0:
        print(f"⚠️ Found {len(negative_prices)} negative prices in {col}, removing...")
        df = df[df[col] >= 0]

print(f"\nCleaned dataset shape: {df.shape}")

# 2. EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
print("\n\n2. EXPLORATORY DATA ANALYSIS")
print("-" * 30)



# Updated statistics

In [ ]:
print("2.1 BASIC STATISTICS")
print("-" * 20)


print("\nCleaned dataset statistics:")
print(df[numeric_cols].describe())

# Date range analysis if date column exists

In [ ]:
if date_cols and not df[date_col].isnull().all():
    print(f"\nTemporal Analysis:")
    print(f"Date Range: {df[date_col].min()} to {df[date_col].max()}")
    print(f"Total Trading Days: {df[date_col].nunique()}")

# Company/Ticker analysis
ticker_cols = [col for col in df.columns if any(term in col.lower()
               for term in ['ticker', 'symbol', 'company', 'stock'])]
if ticker_cols:
    ticker_col = ticker_cols[0]
    print(f"\nStock Analysis:")
    print(f"Unique Stocks/Companies: {df[ticker_col].nunique()}")
    print(f"\nTop 10 by data points:")
    print(df[ticker_col].value_counts().head(10))

# 2.2 DATA VISUALIZATIONS

In [ ]:
print("\n2.2 COMPREHENSIVE DATA VISUALIZATIONS")
print("-" * 40)

# Create comprehensive visualizations

In [ ]:
fig = plt.figure(figsize=(20, 16))


# Identify key columns for visualization

In [ ]:
close_col = None
volume_col = None
open_col = None

for col in df.columns:
    if 'close' in col.lower() and close_col is None:
        close_col = col
    elif 'volume' in col.lower() and volume_col is None:
        volume_col = col
    elif 'open' in col.lower() and open_col is None:
        open_col = col


# 1. Price Distribution

In [ ]:
plt.figure(figsize=(30, 20))
plt.subplot(3, 4, 1)
if close_col:
    plt.hist(df[close_col].dropna(), bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title(f'Distribution of {close_col}')
    plt.xlabel('Price')
    plt.ylabel('Frequency')

# 2. Volume Distribution

In [ ]:
plt.figure(figsize=(25, 10))
plt.subplot(3, 4, 2)
if volume_col:
    volume_data = df[volume_col].dropna()
    if (volume_data > 0).any():
        plt.hist(np.log10(volume_data[volume_data > 0]), bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
        plt.title(f'Distribution of Log10({volume_col})')
        plt.xlabel('Log10(Volume)')
        plt.ylabel('Frequency')

# 3. Correlation heatmap

In [ ]:
plt.figure(figsize=(25, 10))
plt.subplot(3, 4, 3)
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')


# 4. Time series plot

In [ ]:
plt.figure(figsize=(25, 10))
plt.subplot(3, 4, 4)
if date_cols and close_col:
    # Sample data for time series
    if ticker_cols:
        sample_ticker = df[ticker_col].value_counts().index[0]
        sample_data = df[df[ticker_col] == sample_ticker].sort_values(date_col)
    else:
        sample_data = df.sort_values(date_col) if date_col in df.columns else df

    if len(sample_data) > 1:
        plt.plot(sample_data[date_col].iloc[::max(1, len(sample_data)//100)],
                sample_data[close_col].iloc[::max(1, len(sample_data)//100)],
                linewidth=1, alpha=0.8)
        plt.title('Sample Time Series')
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.xticks(rotation=45)


# 5. Additional analysis plots

In [ ]:
plt.figure(figsize=(45, 20))
numeric_cols_subset = numeric_cols[:4] if len(numeric_cols) >= 4 else numeric_cols

for i, col in enumerate(numeric_cols_subset):
    plt.subplot(3, 4, 5 + i)
    plt.hist(df[col].dropna(), bins=30, alpha=0.7, color=plt.cm.Set3(i), edgecolor='black')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')

# 6. Box plots for outlier detection


In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 9)
if len(numeric_cols) >= 2:
    df_sample = df[numeric_cols[:2]].sample(min(1000, len(df)))
    plt.boxplot([df_sample.iloc[:, 0].dropna(), df_sample.iloc[:, 1].dropna()],
                labels=numeric_cols[:2])
    plt.title('Box Plot - Outlier Detection')
    plt.xticks(rotation=45)

# 7. Scatter plot

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 10)
if close_col and volume_col:
    sample_data = df[[close_col, volume_col]].dropna().sample(min(2000, len(df)))
    plt.scatter(sample_data[volume_col], sample_data[close_col], alpha=0.5, s=1)
    plt.title(f'{close_col} vs {volume_col}')
    plt.xlabel(volume_col)
    plt.ylabel(close_col)

# 8. Monthly pattern

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 11)
if date_cols and close_col:
    df['Month'] = df[date_col].dt.month
    monthly_avg = df.groupby('Month')[close_col].mean()
    plt.bar(monthly_avg.index, monthly_avg.values, color='lightcoral', alpha=0.7, edgecolor='black')
    plt.title('Average Price by Month')
    plt.xlabel('Month')
    plt.ylabel('Average Price')

# 9. Statistical summary visualization

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 12)
summary_stats = df[numeric_cols].describe().loc[['mean', 'std', 'min', 'max']]
x_pos = np.arange(len(summary_stats.columns))
plt.bar(x_pos, summary_stats.loc['mean'], alpha=0.7, color='purple', edgecolor='black')
plt.xticks(x_pos, summary_stats.columns, rotation=45)
plt.title('Mean Values by Feature')
plt.ylabel('Mean Value')

plt.tight_layout()
plt.show()

print("✓ Generated comprehensive visualizations")

# 3. FEATURE ENGINEERING

In [ ]:
print("\n\n3. FEATURE ENGINEERING")
print("-" * 25)

print("Creating advanced technical indicators and features...")

# Ensure proper sorting for time series calculations
if date_cols and ticker_cols:
    df = df.sort_values([ticker_col, date_col]).reset_index(drop=True)
elif date_cols:
    df = df.sort_values(date_col).reset_index(drop=True)

# Identify OHLCV columns

In [ ]:
ohlc_mapping = {}
for col in df.columns:
    col_lower = col.lower()
    if 'open' in col_lower:
        ohlc_mapping['open'] = col
    elif 'high' in col_lower:
        ohlc_mapping['high'] = col
    elif 'low' in col_lower:
        ohlc_mapping['low'] = col
    elif 'close' in col_lower:
        ohlc_mapping['close'] = col
    elif 'volume' in col_lower:
        ohlc_mapping['volume'] = col

print(f"Identified OHLCV columns: {ohlc_mapping}")


# Technical Indicators

In [ ]:
if 'high' in ohlc_mapping and 'low' in ohlc_mapping and 'close' in ohlc_mapping:
    # Price-based features
    df['HL_Spread'] = df[ohlc_mapping['high']] - df[ohlc_mapping['low']]
    df['Price_Range'] = df['HL_Spread'] / df[ohlc_mapping['close']]
    df['Volatility'] = (df['HL_Spread'] / df[ohlc_mapping['close']]) * 100

if 'open' in ohlc_mapping and 'close' in ohlc_mapping:
    df['OC_Change'] = df[ohlc_mapping['close']] - df[ohlc_mapping['open']]
    df['Daily_Return'] = (df['OC_Change'] / df[ohlc_mapping['open']]) * 100


# Moving averages

In [ ]:
def safe_rolling_mean(series, window):
    """Safe rolling mean calculation"""
    return series.rolling(window=window, min_periods=1).mean()

if 'close' in ohlc_mapping:
    close_col = ohlc_mapping['close']

    if ticker_cols:
        # Calculate moving averages per ticker
        df['MA_5'] = df.groupby(ticker_col)[close_col].transform(lambda x: safe_rolling_mean(x, 5))
        df['MA_10'] = df.groupby(ticker_col)[close_col].transform(lambda x: safe_rolling_mean(x, 10))
        df['MA_20'] = df.groupby(ticker_col)[close_col].transform(lambda x: safe_rolling_mean(x, 20))
    else:
        # Calculate moving averages for entire dataset
        df['MA_5'] = safe_rolling_mean(df[close_col], 5)
        df['MA_10'] = safe_rolling_mean(df[close_col], 10)
        df['MA_20'] = safe_rolling_mean(df[close_col], 20)

    # Price position relative to moving averages
    df['Price_vs_MA5'] = ((df[close_col] - df['MA_5']) / df['MA_5']) * 100
    df['Price_vs_MA10'] = ((df[close_col] - df['MA_10']) / df['MA_10']) * 100
    df['Price_vs_MA20'] = ((df[close_col] - df['MA_20']) / df['MA_20']) * 100


# Volume indicators

In [ ]:
if 'volume' in ohlc_mapping:
    volume_col = ohlc_mapping['volume']
    if ticker_cols:
        df['Volume_MA5'] = df.groupby(ticker_col)[volume_col].transform(lambda x: safe_rolling_mean(x, 5))
    else:
        df['Volume_MA5'] = safe_rolling_mean(df[volume_col], 5)

    df['Volume_Ratio'] = df[volume_col] / (df['Volume_MA5'] + 1e-8)  # Add small value to avoid division by zero


# Time-based features,Price change features and lag feature

In [ ]:
if date_cols:
    df['DayOfWeek'] = df[date_col].dt.dayofweek
    df['Month'] = df[date_col].dt.month
    df['Quarter'] = df[date_col].dt.quarter
    df['Year'] = df[date_col].dt.year
    df['DayOfMonth'] = df[date_col].dt.day
    df['IsMonthEnd'] = df[date_col].dt.is_month_end.astype(int)

# Lag features
if 'close' in ohlc_mapping:
    if ticker_cols:
        df['Prev_Close'] = df.groupby(ticker_col)[close_col].shift(1)
        df['Prev_2_Close'] = df.groupby(ticker_col)[close_col].shift(2)
    else:
        df['Prev_Close'] = df[close_col].shift(1)
        df['Prev_2_Close'] = df[close_col].shift(2)

    # Price change features
    df['Price_Change'] = ((df[close_col] - df['Prev_Close']) / df['Prev_Close'] * 100).fillna(0)
    df['Price_Change_2d'] = ((df[close_col] - df['Prev_2_Close']) / df['Prev_2_Close'] * 100).fillna(0)


# RSI-like momentum indicator

In [ ]:
if 'close' in ohlc_mapping:
    def calculate_momentum(series, window=14):
        changes = series.diff()
        gains = changes.where(changes > 0, 0)
        losses = -changes.where(changes < 0, 0)
        avg_gains = gains.rolling(window=window, min_periods=1).mean()
        avg_losses = losses.rolling(window=window, min_periods=1).mean()
        rs = avg_gains / (avg_losses + 1e-8)
        momentum = 100 - (100 / (1 + rs))
        return momentum

    if ticker_cols:
        df['Momentum'] = df.groupby(ticker_col)[close_col].transform(lambda x: calculate_momentum(x))
    else:
        df['Momentum'] = calculate_momentum(df[close_col])


# Target variable (next day price change)

In [ ]:
if 'close' in ohlc_mapping:
    if ticker_cols:
        df['Next_Close'] = df.groupby(ticker_col)[close_col].shift(-1)
    else:
        df['Next_Close'] = df[close_col].shift(-1)

    df['Target'] = ((df['Next_Close'] - df[close_col]) / df[close_col] * 100).fillna(0)


# Remove rows with NaN in target for supervised learning

In [ ]:
df_model = df.dropna(subset=['Target'] if 'Target' in df.columns else [])

print("✓ Created comprehensive technical indicators")
print("✓ Generated time-based features")
print("✓ Calculated lag features and momentum indicators")
print("✓ Created target variable for prediction")

print(f"\nFeature Engineering Summary:")
print(f"Features after engineering: {df_model.shape[1]}")
print(f"Records available for modeling: {df_model.shape[0]}")


# Display engineered features

In [ ]:
engineered_features = [col for col in df_model.columns if col not in df.columns[:len(df.columns)-20]]
if engineered_features:
    print(f"\nEngineered features preview:")
    print(df_model[engineered_features].describe())


# 4. FEATURE SELECTION

In [ ]:
print("\n\n4. ADVANCED FEATURE SELECTION")
print("-" * 30)

# Prepare feature matrix
# Exclude non-numeric and target columns
exclude_cols = ['Target', 'Next_Close'] + [col for col in df_model.columns
                                          if df_model[col].dtype == 'object' or 'date' in col.lower()]

feature_candidates = [col for col in df_model.columns if col not in exclude_cols]
print(f"Feature candidates: {len(feature_candidates)}")

# Ensure we have numeric features only

In [ ]:
numeric_features = []
for col in feature_candidates:
    try:
        pd.to_numeric(df_model[col])
        numeric_features.append(col)
    except:
        continue

print(f"Numeric features for selection: {len(numeric_features)}")

# Clean the feature matrix

In [ ]:
X_features = df_model[numeric_features].copy()
y_target = df_model['Target'] if 'Target' in df_model.columns else df_model[close_col]

# Remove any remaining NaN values
mask = ~(X_features.isnull().any(axis=1) | y_target.isnull())
X_clean = X_features[mask]
y_clean = y_target[mask]

print(f"Clean dataset for feature selection: {X_clean.shape}")

# Handle infinite values

In [ ]:
X_clean = X_clean.replace([np.inf, -np.inf], np.nan)
X_clean = X_clean.fillna(X_clean.median())

# 4.1 Correlation-based feature selection

In [ ]:
print("\n4.1 CORRELATION ANALYSIS")
print("-" * 25)

correlations = abs(X_clean.corrwith(y_clean)).sort_values(ascending=False)
print("Top 15 features by correlation with target:")
print(correlations.head(15))


# Remove highly correlated features among predictors

In [ ]:
# Remove highly correlated features
corr_matrix = X_clean.corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_features = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.95)]

if high_corr_features:
    print(f"\nRemoving {len(high_corr_features)} highly correlated features")
    X_clean = X_clean.drop(high_corr_features, axis=1)

# Plot correlation heatmap for top correlated features with the target
target_correlations = X_clean.corr()['Price_Range'].abs().sort_values(ascending=False)
top_features = target_correlations.head(20).index  # Top 20 features including the target
top_corr_matrix = X_clean[top_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(top_corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Top 20 Features (Including Target)')
plt.tight_layout()
plt.show()

# 4.2 Statistical feature selection

In [ ]:
print("\n4.2 STATISTICAL FEATURE SELECTION")
print("-" * 35)

# Select top features using F-test
k_best = min(15, X_clean.shape[1])
selector_f = SelectKBest(score_func=f_regression, k=k_best)

# Handle potential inf values in y_clean before fitting
y_clean = y_clean.replace([np.inf, -np.inf], np.nan)
# You might consider a different imputation strategy for y_clean if NaNs are introduced here
# For now, we'll drop NaNs as SelectKBest doesn't handle them directly in y.
mask_y = y_clean.notnull()
X_clean_masked = X_clean[mask_y]
y_clean_masked = y_clean[mask_y]


X_selected_f = selector_f.fit_transform(X_clean_masked, y_clean_masked)
selected_features_f = X_clean_masked.columns[selector_f.get_support()].tolist()

print(f"Top {k_best} features by F-statistic:")
feature_scores = pd.DataFrame({
    'Feature': X_clean_masked.columns,
    'F_Score': selector_f.scores_
}).sort_values('F_Score', ascending=False)
print(feature_scores.head(k_best))

# 4.3 Recursive Feature Elimination

In [ ]:
# Drop rows where y_clean is NaN
valid_indices = ~y_clean.isna()
X_clean_rfe = X_clean[valid_indices]
y_clean_rfe = y_clean[valid_indices]

# Run RFE
rfe_estimator = LinearRegression()
rfe = RFE(estimator=rfe_estimator, n_features_to_select=12)
rfe.fit(X_clean_rfe, y_clean_rfe)

rfe_selected = X_clean_rfe.columns[rfe.support_].tolist()
print("Features selected by RFE:")
for i, feature in enumerate(rfe_selected, 1):
    print(f"{i:2d}. {feature}")


# 4.4 Final Feature Selection

In [ ]:
print("\n4.5 FINAL FEATURE SELECTION")
print("-" * 30)

# Combine different selection methods
top_corr = correlations.head(10).index.tolist()
top_f_stat = selected_features_f[:10]

top_rfe = rfe_selected[:10]

# Create final feature set (union of top methods)
final_features = list(set(top_corr + top_f_stat  + top_rfe))
final_features = [f for f in final_features if f in X_clean.columns][:15]  # Limit to top 15

print(f"Final selected features ({len(final_features)}):")
for i, feature in enumerate(final_features, 1):
    print(f"{i:2d}. {feature}")

# 5. MODEL BUILDING AND TRAINING

In [ ]:
print("\n\n5. ADVANCED MODEL BUILDING")
print("-" * 28)

# Prepare final datasets
X_final = X_clean[final_features].copy()
y_final = y_clean.copy()

print(f"Final feature matrix shape: {X_final.shape}")
print(f"Target vector shape: {y_final.shape}")

# Train-test split with stratification consideration
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, shuffle=True
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Feature scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled using StandardScaler")

# Initialize advanced models

In [ ]:
print("\n5.1 MODEL INITIALIZATION AND HYPERPARAMETER TUNING")
print("-" * 50)

models = {
    'Linear Regression': LinearRegression(),


    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),

    'SVR': SVR(kernel='rbf', C=100, gamma='scale')
}

# Hyperparameter tuning for key models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, train_test_split
import numpy as np

print("\n5.1 MODEL INITIALIZATION AND HYPERPARAMETER TUNING")
print("-" * 50)

# Initialize the required models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'SVR': SVR()
}

# Define hyperparameter grids
param_grids = {
    'Random Forest': {
        'n_estimators': [50, 100],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5]
    },
    'SVR': {
        'C': [10, 100],
        'gamma': ['scale', 'auto']
    }
}

tuned_models = {}

print("Performing hyperparameter tuning...")

# Handle NaNs in y_train (and align X accordingly)
print(f"Before cleaning: y_train has {y_train.isna().sum()} missing values")
valid_indices = ~y_train.isna()
X_train_scaled = X_train_scaled[valid_indices]
y_train = y_train[valid_indices]
print(f"After cleaning: y_train has {y_train.isna().sum()} missing values")

# Loop through each model
for name, model in models.items():
    print(f"\nProcessing {name}...")

    if name in param_grids:
        print(f"  Tuning {name}...")
        grid_search = GridSearchCV(
            model,
            param_grids[name],
            cv=3,
            scoring='r2',
            n_jobs=-1,
            verbose=1
        )
        grid_search.fit(X_train_scaled, y_train)

        tuned_models[name] = grid_search.best_estimator_
        print(f"  ✓ Best params for {name}: {grid_search.best_params_}")
        print(f"  ✓ Best cross-validation score: {grid_search.best_score_:.4f}")

    else:
        # For Linear Regression (no hyperparameters)
        print(f"  Using default settings for {name} (no tuning grid specified).")
        model.fit(X_train_scaled, y_train)
        tuned_models[name] = model
        print(f"  ✓ {name} trained.")

print("\n✓ Hyperparameter tuning/initial training complete.")
print("\nModels ready for evaluation:")
for name, model in tuned_models.items():
    print(f"- {name}: {model}")


# 5.2 Model Training and Evaluation

In [ ]:
print("\n5.2 MODEL TRAINING AND COMPREHENSIVE EVALUATION")
print("-" * 45)

model_results = {}

for name, model in tuned_models.items():
    print(f"\nTraining and evaluating {name}...")

    # Train model
    start_time = pd.Timestamp.now()
    model.fit(X_train_scaled, y_train)
    training_time = (pd.Timestamp.now() - start_time).total_seconds()

    # Predictions
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)

    # Comprehensive metrics
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    explained_var = explained_variance_score(y_test, y_pred_test)

    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')

    # Store results
    model_results[name] = {
        'model': model,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'explained_variance': explained_var,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'training_time': training_time,
        'predictions': y_pred_test
    }

    print(f"✓ {name} Results:")
    print(f"  Training R²: {train_r2:.4f}")
    print(f"  Test R²: {test_r2:.4f}")
    print(f"  Test RMSE: {test_rmse:.4f}")
    print(f"  Test MAE: {test_mae:.4f}")
    print(f"  CV R² (mean±std): {cv_scores.mean():.4f}±{cv_scores.std():.4f}")
    print(f"  Training time: {training_time:.2f}s")


# 6. COMPREHENSIVE MODEL EVALUATION

In [ ]:
print("\n\n6. COMPREHENSIVE MODEL EVALUATION AND COMPARISON")
print("-" * 50)

# Create detailed comparison DataFrame
comparison_data = []
for name, results in model_results.items():
    comparison_data.append({
        'Model': name,
        'Train_R²': results['train_r2'],
        'Test_R²': results['test_r2'],
        'Train_RMSE': results['train_rmse'],
        'Test_RMSE': results['test_rmse'],
        'Train_MAE': results['train_mae'],
        'Test_MAE': results['test_mae'],
        'CV_R²_Mean': results['cv_mean'],
        'CV_R²_Std': results['cv_std'],
        'Explained_Var': results['explained_variance'],
        'Training_Time': results['training_time'],
        'Overfitting': results['train_r2'] - results['test_r2']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test_R²', ascending=False)

print("COMPREHENSIVE MODEL PERFORMANCE COMPARISON:")
print("=" * 60)
print(comparison_df.round(4).to_string(index=False))


# Identify best model

In [ ]:
# ✅ Correctly identify the best model based on highest Test R² score
best_model_index = comparison_df['Test_R²'].idxmax()
best_model_name = comparison_df.loc[best_model_index, 'Model']
best_model_results = model_results[best_model_name]
best_model = best_model_results['model']

# Display the best model's performance
print(f"\n🏆 BEST PERFORMING MODEL: {best_model_name}")
print("-" * 40)
print(f"Test R² Score: {best_model_results['test_r2']:.4f}")
print(f"Test RMSE: {best_model_results['test_rmse']:.4f}")
print(f"Test MAE: {best_model_results['test_mae']:.4f}")
print(f"Cross-validation R²: {best_model_results['cv_mean']:.4f} ± {best_model_results['cv_std']:.4f}")
print(f"Overfitting measure: {best_model_results['train_r2'] - best_model_results['test_r2']:.4f}")


# Advanced visualizations

In [ ]:
print("\n6.1 ADVANCED MODEL VISUALIZATION")
print("-" * 35)

# Create comprehensive visualization dashboard
fig = plt.figure(figsize=(28, 16))

# 1. Model Performance Comparison
plt.subplot(3, 4, 1)
models_list = comparison_df['Model'].tolist()
test_r2_scores = comparison_df['Test_R²'].tolist()
colors = plt.cm.Set3(np.linspace(0, 1, len(models_list)))

bars = plt.bar(models_list, test_r2_scores, color=colors, alpha=0.8, edgecolor='black')
plt.title('Model Performance Comparison (Test R²)', fontsize=12, fontweight='bold')
plt.ylabel('R² Score')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# 2. Training vs Test Performance

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 2)
train_scores = comparison_df['Train_R²'].tolist()
test_scores = comparison_df['Test_R²'].tolist()

x = np.arange(len(models_list))
width = 0.35

plt.bar(x - width/2, train_scores, width, label='Training R²', alpha=0.8, color='lightblue', edgecolor='black')
plt.bar(x + width/2, test_scores, width, label='Test R²', alpha=0.8, color='lightcoral', edgecolor='black')

plt.title('Training vs Test Performance', fontsize=12, fontweight='bold')
plt.xlabel('Models')
plt.ylabel('R² Score')
plt.xticks(x, models_list, rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3)

# 3. Error Metrics Comparison
plt.subplot(3, 4, 3)
rmse_scores = comparison_df['Test_RMSE'].tolist()
mae_scores = comparison_df['Test_MAE'].tolist()

x = np.arange(len(models_list))
plt.bar(x - width/2, rmse_scores, width, label='RMSE', alpha=0.8, color='lightgreen', edgecolor='black')
plt.bar(x + width/2, mae_scores, width, label='MAE', alpha=0.8, color='orange', edgecolor='black')

plt.title('Error Metrics Comparison', fontsize=12, fontweight='bold')
plt.xlabel('Models')
plt.ylabel('Error Value')
plt.xticks(x, models_list, rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3)


# 3. Cross-validation Performance

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 4)
cv_means = comparison_df['CV_R²_Mean'].tolist()
cv_stds = comparison_df['CV_R²_Std'].tolist()

plt.errorbar(models_list, cv_means, yerr=cv_stds, fmt='o-', capsize=5, capthick=2, linewidth=2, markersize=8)
plt.title('Cross-Validation Performance', fontsize=12, fontweight='bold')
plt.ylabel('CV R² Score')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)


# 4. Actual vs Predicted (Best Model)

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 5)
best_predictions = best_model_results['predictions']
plt.scatter(y_test, best_predictions, alpha=0.6, s=20, color='blue', edgecolors='black', linewidth=0.5)
min_val = min(y_test.min(), best_predictions.min())
max_val = max(y_test.max(), best_predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted - {best_model_name}', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)


# 5. Residual Plot (Best Model)

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 6)
residuals = y_test - best_predictions
plt.scatter(best_predictions, residuals, alpha=0.6, s=20, color='green', edgecolors='black', linewidth=0.5)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title(f'Residual Plot - {best_model_name}', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)

# 6. Training Time Comparison

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 7)
training_times = comparison_df['Training_Time'].tolist()
plt.bar(models_list, training_times, color='purple', alpha=0.7, edgecolor='black')
plt.title('Training Time Comparison', fontsize=12, fontweight='bold')
plt.ylabel('Time (seconds)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# 9. Overfitting Analysis

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 8)
overfitting_scores = comparison_df['Overfitting'].tolist()
colors = ['red' if x > 0.1 else 'orange' if x > 0.05 else 'green' for x in overfitting_scores]
plt.bar(models_list, overfitting_scores, color=colors, alpha=0.7, edgecolor='black')
plt.title('Overfitting Analysis', fontsize=12, fontweight='bold')
plt.ylabel('Train R² - Test R²')
plt.xticks(rotation=45)
plt.axhline(y=0.1, color='red', linestyle='--', alpha=0.7, label='High Overfitting')
plt.axhline(y=0.05, color='orange', linestyle='--', alpha=0.7, label='Moderate Overfitting')
plt.legend()
plt.grid(axis='y', alpha=0.3)

#10.Prediction Distribution

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 10)
plt.hist(y_test, bins=30, alpha=0.7, label='Actual', color='blue', edgecolor='black')
plt.hist(best_predictions, bins=30, alpha=0.7, label='Predicted', color='red', edgecolor='black')
plt.title('Prediction Distribution Comparison', fontsize=12, fontweight='bold')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.legend()
plt.grid(axis='y', alpha=0.3)


# 11. Model Complexity vs Performance

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 11)
model_complexity = {
    'Linear Regression': 1,
    'Ridge Regression': 2,
    'Lasso Regression': 2,
    'Random Forest': 4,
    'Gradient Boosting': 5,
    'SVR': 3
}

complexity_scores = [model_complexity.get(model, 3) for model in models_list]
plt.scatter(complexity_scores, test_r2_scores, s=100, alpha=0.8, color='purple', edgecolors='black', linewidth=2)

for i, model in enumerate(models_list):
    plt.annotate(model, (complexity_scores[i], test_r2_scores[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.title('Model Complexity vs Performance', fontsize=12, fontweight='bold')
plt.xlabel('Complexity Level')
plt.ylabel('Test R² Score')
plt.grid(alpha=0.3)

# 12. Learning Curve Simulation

In [ ]:
plt.figure(figsize=(35, 20))
plt.subplot(3, 4, 12)
# Simulate learning curve for best model
sample_sizes = np.linspace(0.1, 1.0, 10)
train_scores_mean = []
test_scores_mean = []

for size in sample_sizes:
    sample_size = int(len(X_train_scaled) * size)
    if sample_size < 10:
        continue

    X_sample = X_train_scaled[:sample_size]
    y_sample = y_train.iloc[:sample_size]

    # Simple train-validation split
    split_idx = int(len(X_sample) * 0.8)
    X_tr, X_val = X_sample[:split_idx], X_sample[split_idx:]
    y_tr, y_val = y_sample[:split_idx], y_sample[split_idx:]

    if len(X_tr) > 0 and len(X_val) > 0:
        temp_model = type(best_model)(**best_model.get_params())
        temp_model.fit(X_tr, y_tr)

        train_score = r2_score(y_tr, temp_model.predict(X_tr))
        val_score = r2_score(y_val, temp_model.predict(X_val))

        train_scores_mean.append(train_score)
        test_scores_mean.append(val_score)

if len(train_scores_mean) > 0:
    sample_sizes_actual = sample_sizes[:len(train_scores_mean)]
    plt.plot(sample_sizes_actual, train_scores_mean, 'o-', label='Training Score', linewidth=2, markersize=6)
    plt.plot(sample_sizes_actual, test_scores_mean, 'o-', label='Validation Score', linewidth=2, markersize=6)
    plt.title('Learning Curve', fontsize=12, fontweight='bold')
    plt.xlabel('Training Set Size (fraction)')
    plt.ylabel('R² Score')
    plt.legend()
    plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Generated comprehensive model evaluation visualizations")

# Feature importance analysis
if hasattr(best_model, 'feature_importances_'):
    print(f"\n6.2 FEATURE IMPORTANCE ANALYSIS - {best_model_name}")
    print("-" * 50)

    feature_importance_df = pd.DataFrame({
        'Feature': final_features,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)

    print("Top 10 Most Important Features:")
    print(feature_importance_df.head(10).to_string(index=False))

# 7. FINAL RESULTS AND INSIGHTS
print("\n\n7. COMPREHENSIVE RESULTS SUMMARY AND INSIGHTS")
print("-" * 50)

print("🎉 PROJECT COMPLETION SUMMARY")
print("="*60)

# Data processing summary
total_records = len(df_model)
total_features_engineered = len([col for col in df_model.columns if col not in df.columns[:10]])
companies_analyzed = df[ticker_col].nunique() if ticker_cols else "N/A"

print(f"📊 DATA PROCESSING:")
print(f"  • Total records processed: {total_records:,}")
print(f"  • Companies/stocks analyzed: {companies_analyzed}")
print(f"  • Features engineered: {total_features_engineered}")
print(f"  • Final features selected: {len(final_features)}")
print(f"  • Data quality: {((1 - df_model.isnull().sum().sum() / df_model.size) * 100):.1f}% complete")

print(f"\n🤖 MODEL PERFORMANCE:")
print(f"  • Models evaluated: {len(models)}")
print(f"  • Best performing model: {best_model_name}")
print(f"  • Best test R² score: {best_model_results['test_r2']:.4f}")
print(f"  • Best test RMSE: {best_model_results['test_rmse']:.4f}")
print(f"  • Cross-validation score: {best_model_results['cv_mean']:.4f} ± {best_model_results['cv_std']:.4f}")
print(f"  • Model generalization: {'Good' if best_model_results['train_r2'] - best_model_results['test_r2'] < 0.1 else 'Moderate'}")


# Save model artifacts

In [ ]:
print(f"\n💾 SAVING MODEL ARTIFACTS")
print("-" * 30)

try:
    # Save the best model
    joblib.dump(best_model, f'best_model_{best_model_name.lower().replace(" ", "_")}.pkl')
    print(f"✓ Best model saved: best_model_{best_model_name.lower().replace(' ', '_')}.pkl")

    # Save the feature scaler
    joblib.dump(scaler, 'feature_scaler.pkl')
    print("✓ Feature scaler saved: feature_scaler.pkl")

    # Save feature names and metadata
    model_metadata = {
        'best_model_name': best_model_name,
        'feature_names': final_features,
        'model_performance': best_model_results,
        'feature_correlations': correlations.to_dict(),
        'model_comparison': comparison_df.to_dict(),
        'data_summary': {
            'total_records': total_records,
            'companies_analyzed': companies_analyzed,
            'features_engineered': total_features_engineered,
            'date_range': (df[date_col].min(), df[date_col].max()) if date_cols else None
        }
    }

    joblib.dump(model_metadata, 'model_metadata.pkl')
    print("✓ Model metadata saved: model_metadata.pkl")

    # Save the processed feature matrix for potential future use
    feature_data = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_names': final_features
    }
    joblib.dump(feature_data, 'processed_data.pkl')
    print("✓ Processed data saved: processed_data.pkl")

except Exception as e:
    print(f"⚠️ Warning: Could not save model artifacts - {str(e)}")

print("\n" + "="*60)
print("🎉 UK STOCK MARKET ANALYSIS COMPLETED SUCCESSFULLY!")
print("🔬 Advanced Data Mining and Machine Learning Pipeline Executed")
print("📈 Ready for Web Interface Development and Deployment")
print("="*60)

print(f"\n📋 FINAL MODEL PERFORMANCE SUMMARY:")
print(f"Model: {best_model_name}")
print(f"R² Score: {best_model_results['test_r2']:.4f}")
print(f"RMSE: {best_model_results['test_rmse']:.4f}")
print(f"MAE: {best_model_results['test_mae']:.4f}")
print(f"Cross-validation: {best_model_results['cv_mean']:.4f} ± {best_model_results['cv_std']:.4f}")

# Return key objects for GUI development
analysis_results = {
    'best_model': best_model,
    'scaler': scaler,
    'feature_names': final_features,
    'model_results': model_results,
    'comparison_df': comparison_df,
    'processed_data': (X_test, y_test),
    'best_model_name': best_model_name
}

print("\n🖥️  ANALYSIS COMPLETE - READY FOR GUI DEVELOPMENT")
print("="*60)

#GUI

In [ ]:
# UK Stock Market Analysis GUI - Complete Fixed Implementation
# Advanced Data Science Project with Interactive Interface

# Install required packages
!pip install gradio plotly scikit-learn seaborn matplotlib pandas numpy joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# Machine Learning imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE, SelectKBest, f_regression, mutual_info_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
import joblib
import io
import base64
from datetime import datetime

# Set style
plt.style.use('default')
sns.set_palette("husl")

class StockMarketAnalyzer:
    def __init__(self):
        self.df = None
        self.df_processed = None
        self.models = {}
        self.best_model = None
        self.best_model_name = None
        self.scaler = None
        self.feature_names = []
        self.results = {}
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None

    def load_and_preprocess_data(self, file_path):
        """Load and preprocess the uploaded CSV file"""
        try:
            # Read CSV file
            self.df = pd.read_csv(file_path)

            # Basic info
            info = f"✅ Dataset loaded successfully!\n"
            info += f"📊 Shape: {self.df.shape}\n"
            info += f"📈 Records: {self.df.shape[0]:,}\n"
            info += f"🔢 Columns: {self.df.shape[1]}\n\n"

            # Show original columns
            info += f"📋 Original Columns: {list(self.df.columns)}\n\n"

            # Optimize dataset size if too large
            if self.df.shape[0] > 2000:
                self.df = self.df.sample(min(2000, self.df.shape[0]), random_state=42)
                info += f"⚡ Optimized to {self.df.shape[0]:,} records for efficiency\n\n"

            # Data cleaning
            info += self._clean_data()

            return info, self.df.head(), self.df.describe()

        except Exception as e:
            return f"❌ Error loading data: {str(e)}", None, None

    def _clean_data(self):
        """Clean and preprocess the data"""
        info = "🧹 DATA CLEANING:\n"

        # Handle index columns
        index_cols = [col for col in self.df.columns if 'unnamed' in col.lower() or col.lower() in ['index', 'id']]
        if index_cols:
            self.df = self.df.drop(index_cols, axis=1)
            info += f"  ✓ Dropped index columns: {index_cols}\n"

        # Standardize column names
        self.df.columns = self.df.columns.str.strip().str.replace(' ', '_').str.replace('.', '_')
        info += "  ✓ Standardized column names\n"

        # Handle date columns
        date_cols = [col for col in self.df.columns if 'date' in col.lower() or 'time' in col.lower()]
        if date_cols:
            date_col = date_cols[0]
            try:
                self.df[date_col] = pd.to_datetime(self.df[date_col], errors='coerce')
                info += f"  ✓ Converted {date_col} to datetime\n"
            except:
                info += f"  ⚠️ Could not convert {date_col} to datetime\n"

        # Handle missing values
        missing_before = self.df.isnull().sum().sum()
        if missing_before > 0:
            numeric_cols = self.df.select_dtypes(include=[np.number]).columns
            for col in numeric_cols:
                if self.df[col].isnull().sum() > 0:
                    self.df[col] = self.df[col].fillna(self.df[col].median())

            categorical_cols = self.df.select_dtypes(include=['object']).columns
            for col in categorical_cols:
                if self.df[col].isnull().sum() > 0:
                    mode_val = self.df[col].mode()
                    self.df[col] = self.df[col].fillna(mode_val[0] if len(mode_val) > 0 else 'Unknown')

            info += f"  ✓ Handled {missing_before} missing values\n"

        # Remove duplicates
        initial_shape = self.df.shape[0]
        self.df = self.df.drop_duplicates()
        info += f"  ✓ Removed {initial_shape - self.df.shape[0]} duplicates\n"

        return info

    def create_eda_plots(self):
        """Create comprehensive EDA visualizations"""
        if self.df is None:
            return "❌ Please load data first"

        try:
            # Identify key columns
            numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
            close_col = None
            volume_col = None
            date_col = None
            ticker_col = None

            for col in self.df.columns:
                col_lower = col.lower()
                if 'close' in col_lower and close_col is None:
                    close_col = col
                elif 'volume' in col_lower and volume_col is None:
                    volume_col = col
                elif 'date' in col_lower and date_col is None:
                    date_col = col
                elif any(term in col_lower for term in ['ticker', 'symbol', 'company']) and ticker_col is None:
                    ticker_col = col

            # Create subplots
            fig = make_subplots(
                rows=2, cols=2,
                subplot_titles=['Price Distribution', 'Volume Distribution',
                              'Correlation Heatmap', 'Time Series Sample'],
                specs=[[{"type": "histogram"}, {"type": "histogram"}],
                       [{"type": "heatmap"}, {"type": "scatter"}]]
            )

            # 1. Price Distribution
            if close_col and close_col in self.df.columns:
                price_data = self.df[close_col].dropna()
                if len(price_data) > 0:
                    fig.add_trace(
                        go.Histogram(x=price_data, name="Price Distribution", nbinsx=30),
                        row=1, col=1
                    )

            # 2. Volume Distribution
            if volume_col and volume_col in self.df.columns:
                volume_data = self.df[volume_col].dropna()
                volume_data = volume_data[volume_data > 0]
                if len(volume_data) > 0:
                    fig.add_trace(
                        go.Histogram(x=volume_data, name="Volume Distribution", nbinsx=30),
                        row=1, col=2
                    )

            # 3. Correlation Heatmap
            if len(numeric_cols) > 1:
                corr_data = self.df[numeric_cols[:8]].corr()  # Limit for visibility
                fig.add_trace(
                    go.Heatmap(z=corr_data.values,
                              x=corr_data.columns,
                              y=corr_data.columns,
                              colorscale='RdBu', zmid=0),
                    row=2, col=1
                )

            # 4. Time Series Sample
            if date_col and close_col and date_col in self.df.columns and close_col in self.df.columns:
                try:
                    sample_data = self.df[[date_col, close_col]].dropna()
                    if len(sample_data) > 0:
                        sample_data = sample_data.sort_values(date_col)
                        # Sample every nth point for performance
                        n = max(1, len(sample_data) // 100)
                        sample_data = sample_data.iloc[::n]

                        fig.add_trace(
                            go.Scatter(x=sample_data[date_col], y=sample_data[close_col],
                                      mode='lines', name='Price Time Series'),
                            row=2, col=2
                        )
                except:
                    pass

            # Update layout
            fig.update_layout(height=800, showlegend=False, title_text="Exploratory Data Analysis Dashboard")
            return fig

        except Exception as e:
            return f"❌ Error creating EDA plots: {str(e)}"

    def engineer_features(self):
        """Create technical indicators and features"""
        if self.df is None:
            return "❌ Please load data first", None

        try:
            info = "🔧 FEATURE ENGINEERING:\n"
            self.df_processed = self.df.copy()

            # Identify columns
            ohlc_mapping = {}
            date_col = None
            ticker_col = None

            for col in self.df_processed.columns:
                col_lower = col.lower()
                if 'open' in col_lower and 'open' not in ohlc_mapping:
                    ohlc_mapping['open'] = col
                elif 'high' in col_lower and 'high' not in ohlc_mapping:
                    ohlc_mapping['high'] = col
                elif 'low' in col_lower and 'low' not in ohlc_mapping:
                    ohlc_mapping['low'] = col
                elif 'close' in col_lower and 'close' not in ohlc_mapping:
                    ohlc_mapping['close'] = col
                elif 'volume' in col_lower and 'volume' not in ohlc_mapping:
                    ohlc_mapping['volume'] = col
                elif 'date' in col_lower and date_col is None:
                    date_col = col
                elif any(term in col_lower for term in ['ticker', 'symbol', 'company']) and ticker_col is None:
                    ticker_col = col

            info += f"  ✓ Identified columns: {list(ohlc_mapping.keys())}\n"

            # Sort data if possible
            if date_col and date_col in self.df_processed.columns:
                try:
                    if ticker_col and ticker_col in self.df_processed.columns:
                        self.df_processed = self.df_processed.sort_values([ticker_col, date_col]).reset_index(drop=True)
                    else:
                        self.df_processed = self.df_processed.sort_values(date_col).reset_index(drop=True)
                except:
                    pass

            features_created = 0

            # Price-based features
            if 'high' in ohlc_mapping and 'low' in ohlc_mapping and 'close' in ohlc_mapping:
                try:
                    high_col = ohlc_mapping['high']
                    low_col = ohlc_mapping['low']
                    close_col = ohlc_mapping['close']

                    self.df_processed['HL_Spread'] = self.df_processed[high_col] - self.df_processed[low_col]
                    self.df_processed['Price_Range'] = self.df_processed['HL_Spread'] / (self.df_processed[close_col] + 1e-8)
                    self.df_processed['Volatility'] = (self.df_processed['HL_Spread'] / (self.df_processed[close_col] + 1e-8)) * 100
                    features_created += 3
                except:
                    pass

            if 'open' in ohlc_mapping and 'close' in ohlc_mapping:
                try:
                    open_col = ohlc_mapping['open']
                    close_col = ohlc_mapping['close']

                    self.df_processed['OC_Change'] = self.df_processed[close_col] - self.df_processed[open_col]
                    self.df_processed['Daily_Return'] = (self.df_processed['OC_Change'] / (self.df_processed[open_col] + 1e-8)) * 100
                    features_created += 2
                except:
                    pass

            # Moving averages
            if 'close' in ohlc_mapping:
                try:
                    close_col = ohlc_mapping['close']

                    def safe_rolling_mean(series, window):
                        return series.rolling(window=window, min_periods=1).mean()

                    if ticker_col and ticker_col in self.df_processed.columns:
                        self.df_processed['MA_5'] = self.df_processed.groupby(ticker_col)[close_col].transform(lambda x: safe_rolling_mean(x, 5))
                        self.df_processed['MA_10'] = self.df_processed.groupby(ticker_col)[close_col].transform(lambda x: safe_rolling_mean(x, 10))
                    else:
                        self.df_processed['MA_5'] = safe_rolling_mean(self.df_processed[close_col], 5)
                        self.df_processed['MA_10'] = safe_rolling_mean(self.df_processed[close_col], 10)

                    # Price position relative to MAs
                    self.df_processed['Price_vs_MA5'] = ((self.df_processed[close_col] - self.df_processed['MA_5']) / (self.df_processed['MA_5'] + 1e-8)) * 100
                    self.df_processed['Price_vs_MA10'] = ((self.df_processed[close_col] - self.df_processed['MA_10']) / (self.df_processed['MA_10'] + 1e-8)) * 100
                    features_created += 4
                except:
                    pass

            # Volume features
            if 'volume' in ohlc_mapping:
                try:
                    volume_col = ohlc_mapping['volume']

                    def safe_rolling_mean(series, window):
                        return series.rolling(window=window, min_periods=1).mean()

                    if ticker_col and ticker_col in self.df_processed.columns:
                        self.df_processed['Volume_MA5'] = self.df_processed.groupby(ticker_col)[volume_col].transform(lambda x: safe_rolling_mean(x, 5))
                    else:
                        self.df_processed['Volume_MA5'] = safe_rolling_mean(self.df_processed[volume_col], 5)

                    self.df_processed['Volume_Ratio'] = self.df_processed[volume_col] / (self.df_processed['Volume_MA5'] + 1e-8)
                    features_created += 2
                except:
                    pass

            # Time-based features
            if date_col and date_col in self.df_processed.columns:
                try:
                    date_series = pd.to_datetime(self.df_processed[date_col], errors='coerce')
                    self.df_processed['DayOfWeek'] = date_series.dt.dayofweek
                    self.df_processed['Month'] = date_series.dt.month
                    self.df_processed['Quarter'] = date_series.dt.quarter
                    self.df_processed['DayOfMonth'] = date_series.dt.day
                    features_created += 4
                except:
                    pass

            # Create target variable
            if 'close' in ohlc_mapping:
                try:
                    close_col = ohlc_mapping['close']
                    if ticker_col and ticker_col in self.df_processed.columns:
                        self.df_processed['Next_Close'] = self.df_processed.groupby(ticker_col)[close_col].shift(-1)
                    else:
                        self.df_processed['Next_Close'] = self.df_processed[close_col].shift(-1)

                    self.df_processed['Target'] = ((self.df_processed['Next_Close'] - self.df_processed[close_col]) / (self.df_processed[close_col] + 1e-8) * 100)
                    features_created += 1
                except:
                    pass

            # Clean the processed dataframe
            self.df_processed = self.df_processed.replace([np.inf, -np.inf], np.nan)
            self.df_processed = self.df_processed.dropna(subset=['Target'] if 'Target' in self.df_processed.columns else [])

            info += f"  ✓ Created {features_created} technical indicators\n"
            info += f"  ✓ Final dataset shape: {self.df_processed.shape}\n"
            info += f"  ✓ Records available for modeling: {self.df_processed.shape[0]:,}\n"

            return info, self.df_processed.describe()

        except Exception as e:
            return f"❌ Error in feature engineering: {str(e)}", None

    def perform_feature_selection(self):
        """Advanced feature selection methods"""
        if self.df_processed is None:
            return "❌ Please engineer features first", None

        try:
            info = "🎯 FEATURE SELECTION:\n"

            # Prepare feature matrix
            exclude_cols = {'Target', 'Next_Close'}

            # Add date and categorical columns to exclude
            for col in self.df_processed.columns:
                if (self.df_processed[col].dtype == 'object' or
                    'date' in col.lower() or
                    col in exclude_cols):
                    exclude_cols.add(col)

            feature_candidates = [col for col in self.df_processed.columns if col not in exclude_cols]

            # Ensure numeric features only
            numeric_features = []
            for col in feature_candidates:
                try:
                    if self.df_processed[col].dtype in ['int64', 'float64']:
                        numeric_features.append(col)
                except:
                    continue

            info += f"  📊 Feature candidates: {len(numeric_features)}\n"

            if len(numeric_features) == 0:
                return "❌ No numeric features found for selection", None

            # Prepare clean data
            X_features = self.df_processed[numeric_features].copy()

            # Use Target if available, otherwise use first numeric column
            if 'Target' in self.df_processed.columns:
                y_target = self.df_processed['Target'].copy()
            else:
                y_target = X_features.iloc[:, 0].copy()
                X_features = X_features.iloc[:, 1:]
                numeric_features = numeric_features[1:]

            # Remove NaN and infinite values
            mask = ~(X_features.isnull().any(axis=1) | y_target.isnull())
            X_clean = X_features[mask].copy()
            y_clean = y_target[mask].copy()

            X_clean = X_clean.replace([np.inf, -np.inf], np.nan)
            X_clean = X_clean.fillna(X_clean.median())
            y_clean = y_clean.replace([np.inf, -np.inf], np.nan)
            y_clean = y_clean.fillna(y_clean.median())

            info += f"  🧹 Clean dataset: {X_clean.shape}\n"

            if X_clean.shape[0] < 10:
                return "❌ Insufficient data for feature selection", None

            # Correlation analysis
            try:
                correlations = abs(X_clean.corrwith(y_clean)).sort_values(ascending=False)
                correlations = correlations.dropna()
            except:
                correlations = pd.Series(index=X_clean.columns, data=0.1)

            # Remove highly correlated features
            try:
                corr_matrix = X_clean.corr().abs()
                upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                high_corr_features = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.95)]

                if high_corr_features:
                    X_clean = X_clean.drop(high_corr_features, axis=1)
                    info += f"  🔄 Removed {len(high_corr_features)} highly correlated features\n"
            except:
                pass

            # Feature selection methods
            k_best = min(12, X_clean.shape[1])

            selected_features = []

            # F-test selection
            try:
                selector_f = SelectKBest(score_func=f_regression, k=k_best)
                selector_f.fit(X_clean, y_clean)
                selected_features_f = X_clean.columns[selector_f.get_support()].tolist()
                selected_features.extend(selected_features_f[:8])
            except:
                pass

            # Correlation-based selection
            top_corr = correlations.head(8).index.tolist()
            selected_features.extend(top_corr)

            # Remove duplicates and limit features
            self.feature_names = list(dict.fromkeys(selected_features))[:15]  # Remove duplicates while preserving order
            self.feature_names = [f for f in self.feature_names if f in X_clean.columns]

            if len(self.feature_names) == 0:
                self.feature_names = X_clean.columns[:8].tolist()

            info += f"  🎯 Final selected features: {len(self.feature_names)}\n"
            for i, feature in enumerate(self.feature_names, 1):
                info += f"    {i:2d}. {feature}\n"

            # Create correlation plot
            fig = go.Figure()
            if len(correlations) > 0:
                top_features = correlations.head(min(15, len(correlations)))
                fig.add_trace(go.Bar(
                    x=top_features.values,
                    y=top_features.index,
                    orientation='h',
                    name='Correlation with Target'
                ))

            fig.update_layout(
                title='Feature Correlation with Target',
                xaxis_title='Absolute Correlation',
                height=600
            )

            return info, fig

        except Exception as e:
            return f"❌ Error in feature selection: {str(e)}", None

    def train_models(self):
        """Train and evaluate multiple ML models"""
        if not self.feature_names:
            return "❌ Please perform feature selection first", None, None

        try:
            info = "🤖 MODEL TRAINING:\n"

            # Prepare final datasets
            X_final = self.df_processed[self.feature_names].copy()

            if 'Target' in self.df_processed.columns:
                y_final = self.df_processed['Target'].copy()
            else:
                return "❌ Target variable not found", None, None

            # Clean data
            mask = ~(X_final.isnull().any(axis=1) | y_final.isnull())
            X_final = X_final[mask]
            y_final = y_final[mask]

            X_final = X_final.replace([np.inf, -np.inf], np.nan)
            X_final = X_final.fillna(X_final.median())
            y_final = y_final.replace([np.inf, -np.inf], np.nan)
            y_final = y_final.fillna(y_final.median())

            info += f"  📊 Training data shape: {X_final.shape}\n"

            if X_final.shape[0] < 20:
                return "❌ Insufficient data for model training", None, None

            # Train-test split
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                X_final, y_final, test_size=0.2, random_state=42, shuffle=True
            )

            # Feature scaling
            self.scaler = StandardScaler()
            X_train_scaled = self.scaler.fit_transform(self.X_train)
            X_test_scaled = self.scaler.transform(self.X_test)

            # Initialize models
            models = {
                'Linear Regression': LinearRegression(),

                'Random Forest': RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42),
               'SVR': SVR(kernel='rbf', C=100, gamma='scale')
            }

            # Train and evaluate models
            comparison_data = []
            self.models = {}

            for name, model in models.items():
                try:
                    info += f"  🔄 Training {name}...\n"

                    # Train model
                    model.fit(X_train_scaled, self.y_train)

                    # Predictions
                    y_pred_train = model.predict(X_train_scaled)
                    y_pred_test = model.predict(X_test_scaled)

                    # Metrics
                    train_r2 = r2_score(self.y_train, y_pred_train)
                    test_r2 = r2_score(self.y_test, y_pred_test)
                    test_rmse = np.sqrt(mean_squared_error(self.y_test, y_pred_test))
                    test_mae = mean_absolute_error(self.y_test, y_pred_test)

                    # Cross-validation
                    try:
                        cv_scores = cross_val_score(model, X_train_scaled, self.y_train, cv=3, scoring='r2')
                        cv_mean = cv_scores.mean()
                        cv_std = cv_scores.std()
                    except:
                        cv_mean = test_r2
                        cv_std = 0.0

                    # Store results
                    self.models[name] = {
                        'model': model,
                        'train_r2': train_r2,
                        'test_r2': test_r2,
                        'test_rmse': test_rmse,
                        'test_mae': test_mae,
                        'cv_mean': cv_mean,
                        'cv_std': cv_std,
                        'predictions': y_pred_test,
                        'y_test': self.y_test
                    }

                    comparison_data.append({
                        'Model': name,
                        'Train_R²': round(train_r2, 4),
                        'Test_R²': round(test_r2, 4),
                        'Test_RMSE': round(test_rmse, 4),
                        'Test_MAE': round(test_mae, 4),
                        'CV_R²_Mean': round(cv_mean, 4),
                        'CV_R²_Std': round(cv_std, 4),
                        'Overfitting': round(train_r2 - test_r2, 4)
                    })

                    info += f"    ✓ Test R²: {test_r2:.4f}, RMSE: {test_rmse:.4f}\n"

                except Exception as e:
                    info += f"    ❌ Error training {name}: {str(e)}\n"
                    continue

            if not comparison_data:
                return "❌ No models trained successfully", None, None

            # Create comparison DataFrame
            comparison_df = pd.DataFrame(comparison_data)
            comparison_df = comparison_df.sort_values('Test_R²', ascending=False)

            # Identify best model
            self.best_model_name = comparison_df.iloc[0]['Model']
            self.best_model = self.models[self.best_model_name]['model']

            info += f"\n  🏆 Best Model: {self.best_model_name}\n"
            info += f"    📈 Test R²: {comparison_df.iloc[0]['Test_R²']:.4f}\n"
            info += f"    📉 Test RMSE: {comparison_df.iloc[0]['Test_RMSE']:.4f}\n"

            # Create visualization
            fig = make_subplots(
                rows=2, cols=2,
                subplot_titles=['Model Performance Comparison', 'Training vs Test R²',
                              'Actual vs Predicted (Best Model)', 'Residual Plot (Best Model)']
            )

            # Model comparison
            fig.add_trace(
                go.Bar(x=comparison_df['Model'], y=comparison_df['Test_R²'], name='Test R²'),
                row=1, col=1
            )

            # Training vs Test
            fig.add_trace(
                go.Bar(x=comparison_df['Model'], y=comparison_df['Train_R²'], name='Train R²', opacity=0.7),
                row=1, col=2
            )
            fig.add_trace(
                go.Bar(x=comparison_df['Model'], y=comparison_df['Test_R²'], name='Test R²', opacity=0.7),
                row=1, col=2
            )

            # Actual vs Predicted
            if self.best_model_name in self.models:
                best_results = self.models[self.best_model_name]
                fig.add_trace(
                    go.Scatter(x=best_results['y_test'], y=best_results['predictions'],
                              mode='markers', name='Predictions', opacity=0.6),
                    row=2, col=1
                )

                # Perfect prediction line
                y_test_vals = best_results['y_test'].values if hasattr(best_results['y_test'], 'values') else best_results['y_test']
                pred_vals = best_results['predictions']
                min_val = min(np.min(y_test_vals), np.min(pred_vals))
                max_val = max(np.max(y_test_vals), np.max(pred_vals))

                fig.add_trace(
                    go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
                              mode='lines', name='Perfect Prediction', line=dict(dash='dash')),
                    row=2, col=1
                )

                # Residuals
                residuals = y_test_vals -pred_vals

                fig.add_trace(
                    go.Scatter(x=pred_vals, y=residuals,
                              mode='markers', name='Residuals', opacity=0.6),
                    row=2, col=2
                )
                fig.add_hline(y=0, line_dash="dash", line_color="red", row=2, col=2)


            # Update layout
            fig.update_layout(height=1000, showlegend=True, title_text="Model Training and Evaluation Dashboard")

            return info, comparison_df.round(4), fig

        except Exception as e:
            return f"❌ Error training models: {str(e)}", None, None

    def get_model_predictions(self):
        """Return test set predictions and actual values for the best model"""
        if self.best_model is None:
            return "❌ No model has been trained yet."

        try:
            best_results = self.models.get(self.best_model_name)
            if best_results:
                 # Ensure y_test is a Series or numpy array for consistent indexing/values
                y_test_vals = best_results['y_test'].values if hasattr(best_results['y_test'], 'values') else best_results['y_test']
                pred_vals = best_results['predictions']

                # Create a DataFrame for display
                prediction_df = pd.DataFrame({'Actual': y_test_vals, 'Predicted': pred_vals})
                return prediction_df.round(4)
            else:
                return "❌ Results for the best model not found."

        except Exception as e:
            return f"❌ Error retrieving predictions: {str(e)}"

    def save_best_model(self):
        """Save the best performing model and scaler"""
        if self.best_model is None or self.scaler is None or not self.feature_names:
            return "❌ Model, scaler, or feature names are not available to save."

        try:
            # Define filenames
            model_filename = f"best_model_{self.best_model_name.lower().replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
            scaler_filename = f"feature_scaler_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
            features_filename = f"feature_names_{datetime.now().strftime('%Y%m%d_%H%M%S')}.joblib"


            # Save using joblib
            joblib.dump(self.best_model, model_filename)
            joblib.dump(self.scaler, scaler_filename)
            joblib.dump(self.feature_names, features_filename)

            return f"✅ Best model ({self.best_model_name}), scaler, and feature names saved successfully:\n- {model_filename}\n- {scaler_filename}\n- {features_filename}"

        except Exception as e:
            return f"❌ Error saving model artifacts: {str(e)}"


# Instantiate the analyzer class
analyzer = StockMarketAnalyzer()


# Define the Gradio interface blocks
with gr.Blocks() as demo:
    gr.Markdown("# UK Stock Market Analysis & Prediction")
    gr.Markdown("Upload your UK stock market data (CSV format) to perform analysis, feature engineering, model training, and prediction.")

    with gr.Tab("1. Load & Preprocess Data"):
        file_upload = gr.File(label="Upload CSV File")
        load_button = gr.Button("Load & Preprocess")
        load_output_info = gr.Textbox(label="Loading & Preprocessing Info", lines=5)
        load_output_head = gr.DataFrame(label="Dataset Head")
        load_output_desc = gr.DataFrame(label="Dataset Description")

    with gr.Tab("2. Exploratory Data Analysis (EDA)"):
        eda_button = gr.Button("Generate EDA Plots")
        eda_output_plot = gr.Plot(label="EDA Visualizations")

    with gr.Tab("3. Feature Engineering"):
        fe_button = gr.Button("Perform Feature Engineering")
        fe_output_info = gr.Textbox(label="Feature Engineering Info", lines=5)
        fe_output_desc = gr.DataFrame(label="Engineered Features Description")

    with gr.Tab("4. Feature Selection"):
        fs_button = gr.Button("Perform Feature Selection")
        fs_output_info = gr.Textbox(label="Feature Selection Info", lines=5)
        fs_output_plot = gr.Plot(label="Feature Correlation Plot")


    with gr.Tab("5. Model Training & Evaluation"):
        train_button = gr.Button("Train & Evaluate Models")
        train_output_info = gr.Textbox(label="Training Info", lines=5)
        train_output_comparison = gr.DataFrame(label="Model Performance Comparison")
        train_output_plot = gr.Plot(label="Model Evaluation Plots")

    with gr.Tab("6. View Predictions & Save Model"):
        view_pred_button = gr.Button("View Best Model Predictions (Test Set)")
        predictions_output = gr.DataFrame(label="Actual vs Predicted Values")
        save_model_button = gr.Button("Save Best Model Artifacts (.pkl, .joblib)")
        save_model_output = gr.Textbox(label="Save Model Status")


    # Connect components with functions
    load_button.click(
        analyzer.load_and_preprocess_data,
        inputs=file_upload,
        outputs=[load_output_info, load_output_head, load_output_desc]
    )

    eda_button.click(
        analyzer.create_eda_plots,
        inputs=None,
        outputs=eda_output_plot
    )

    fe_button.click(
        analyzer.engineer_features,
        inputs=None,
        outputs=[fe_output_info, fe_output_desc]
    )

    fs_button.click(
        analyzer.perform_feature_selection,
        inputs=None,
        outputs=[fs_output_info, fs_output_plot]
    )

    train_button.click(
        analyzer.train_models,
        inputs=None,
        outputs=[train_output_info, train_output_comparison, train_output_plot]
    )

    view_pred_button.click(
        analyzer.get_model_predictions,
        inputs=None,
        outputs=predictions_output
    )

    save_model_button.click(
        analyzer.save_best_model,
        inputs=None,
        outputs=save_model_output
    )


# Launch the Gradio app
if __name__ == "__main__":
    demo.launch(debug=True, share=True)